# Vision Models

In 2026, the industry standard for the "detect → mask → modify" workflow is built on SAM 2 (Segment Anything Model 2) for masking and Generative Fill (Diffusion Models) for removal or replacement.

pipeline that:
- Detects an object using a prompt (like "the coffee cup").
- Generates a binary mask using SAM 2.
- Removes the object using "Inpainting" (filling the mask with background pixels).

Image → [Detection/Segmentation Model] → Mask → [Inpainting Model] → Result

**1. Object Detection with YOLO**

Task: Find bounding boxes around objects in an image. What it does: Draws labeled bounding boxes. YOLO is fast (real-time capable) and works on 80 common object classes (COCO dataset).

In [3]:
# pip install ultralytics
from ultralytics import YOLO
from PIL import Image, ImageDraw

model = YOLO("yolov8n.pt")  # auto-downloads nano model (~6MB)
image_path = "./data/pic.jpg"

results = model(image_path)[0]

# Draw boxes on image
img = Image.open(image_path)
draw = ImageDraw.Draw(img)

for box in results.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cls = results.names[int(box.cls)]
    conf = float(box.conf)
    draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    draw.text((x1, y1 - 12), f"{cls} {conf:.2f}", fill="red")

img.save("./data/detected.jpg")


image 1/1 c:\personal\genai\gen-ai\new\data\pic.jpg: 384x640 1 bird, 1 cat, 1 dog, 110.0ms
Speed: 4.1ms preprocess, 110.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


This piece of code demonstrates a complete pipeline for visual understanding using a modern deep learning model, specifically YOLOv8. While it may look compact, it encapsulates a powerful idea at the core of computer vision: enabling a machine to interpret and annotate the contents of an image in a way that resembles human perception.

At the beginning, the code loads a pretrained model using the ultralytics library. The line that initializes YOLO("yolov8n.pt") is doing more than just loading weights; it is instantiating a neural network that has already learned to recognize dozens of object categories from large datasets such as COCO. The suffix “n” refers to the nano version of the model, which is intentionally lightweight and optimized for speed, making it suitable for real-time or resource-constrained environments.

Once the model is initialized, it is applied directly to an image path. This is an important design choice in the Ultralytics API, as it abstracts away preprocessing steps such as resizing, normalization, and tensor conversion. Internally, the image is transformed into a format suitable for the neural network, passed through multiple convolutional layers, and processed to produce predictions about object locations and classes.

The output of this inference step is stored in a structure called results. Conceptually, this object contains all the information the model inferred about the image. When indexing with [0], the code extracts the result corresponding to the single input image. Inside this result, one of the most important attributes is boxes, which represents the detected objects in terms of bounding boxes.

Each bounding box is defined by four coordinates: the top-left corner and the bottom-right corner. These coordinates are stored in a tensor format, and the code converts them into integers so they can be used for drawing. Alongside the spatial information, each box also carries semantic information: the predicted class label and a confidence score. The class label is an index that is mapped to a human-readable name using the results.names dictionary, while the confidence score reflects how certain the model is about its prediction.

The next part of the code shifts from inference to visualization. The original image is opened using the Python Imaging Library, and a drawing context is created. For each detected object, a rectangle is drawn around the predicted location. This visual representation is essential for interpreting the model’s output, as it allows us to verify whether the detections align with the actual objects in the image.

In addition to drawing rectangles, the code annotates each detection with text. This text includes both the class name and the confidence score, formatted to two decimal places. The positioning of the text slightly above the bounding box ensures that it does not obscure the object itself, although in practice this can still be an issue if objects are close together or near the image boundary.

Finally, the annotated image is saved to disk. At this point, the pipeline has transformed a raw image into an enriched representation that contains both visual and semantic information. The output image is no longer just a collection of pixels; it becomes a structured depiction of the scene, highlighting what objects are present and where they are located.

It is worth noting that while this example is often associated with “generative AI,” the model used here is technically a discriminative model rather than a generative one. Its purpose is to analyze and label existing data rather than to create new content. However, it plays a crucial role in broader generative pipelines. For instance, object detection models are frequently used as a first step in systems that perform image editing, inpainting, or scene manipulation, where understanding the structure of the image is necessary before generating modifications.

From a broader perspective, this code illustrates a key paradigm in modern AI systems: the combination of learned representations and simple post-processing to achieve meaningful results. The neural network handles the complex task of perception, while the surrounding Python code translates its outputs into a form that humans can easily interpret and use.

**2. Segmentation Mask with SAM (Segment Anything Model)**

Task: Produce a pixel-perfect mask for any object — either by clicking a point or specifying a box.

SAM can segment any object with just a point or box — no class labels needed. It's the industry standard for zero-shot segmentation.

In [ ]:
# pip install segment-anything torch torchvision
# Download checkpoint: wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

import numpy as np
import torch
from PIL import Image
from segment_anything import sam_model_registry, SamPredictor

# Load model
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to("cuda" if torch.cuda.is_available() else "cpu")
predictor = SamPredictor(sam)

# Load image
img = np.array(Image.open("./data/pic.jpg").convert("RGB"))
predictor.set_image(img)

# Prompt: click a point on the object you want to segment
# (x, y) pixel coordinate, label=1 means "foreground"
input_point = np.array([[190, 190]])
input_label = np.array([1])

masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,  # returns 3 mask candidates
)

# Best mask
best_mask = masks[np.argmax(scores)]  # shape: (H, W), dtype: bool

# Save mask as image
mask_img = Image.fromarray((best_mask * 255).astype(np.uint8))
mask_img.save("./data/mask.png")

This code demonstrates a fundamentally different paradigm of visual intelligence compared to classical object detection. Instead of asking the model to identify predefined categories, it allows a user to interactively specify what should be extracted from an image. The core technology behind this is the Segment Anything Model (SAM), a powerful vision model designed to generalize segmentation across arbitrary objects without requiring task-specific retraining.

The process begins with loading a pretrained model using a registry mechanism. When the code calls sam_model_registry["vit_b"], it selects a specific architecture variant of SAM based on a Vision Transformer backbone. The "vit_b" configuration represents a balance between computational efficiency and segmentation quality. The checkpoint file sam_vit_b_01ec64.pth contains the learned parameters of this network, which encode a vast amount of visual knowledge acquired during training on large-scale datasets.

Once the model is initialized, it is moved to an available computation device. If a GPU is present, the model is transferred to CUDA memory to accelerate inference. Otherwise, it falls back to the CPU. This step is essential because segmentation models like SAM are computationally intensive, and performance differences between CPU and GPU execution can be significant.

The SamPredictor object acts as a high-level interface that manages the interaction between the raw image and the model. When the image is loaded and passed to predictor.set_image, the model computes an internal representation, often referred to as an embedding. This embedding is a dense numerical encoding of the visual content of the image, capturing structure, textures, and semantic cues. Importantly, this computation happens only once per image, enabling efficient reuse when multiple prompts are applied.

The next stage introduces the concept of prompting, which is central to SAM’s design. Instead of providing a textual description, the model is guided through spatial input. The variable input_point represents a coordinate in the image, interpreted as a user click. The corresponding label indicates whether this point belongs to the foreground or the background. In this case, a label of 1 signals that the selected point lies on the object of interest. This simple interaction effectively replaces the need for manual annotation or predefined class labels.

When the predict function is called, the model uses both the image embedding and the prompt to generate segmentation masks. Internally, SAM evaluates multiple plausible interpretations of the prompt, which is why it returns several candidate masks when multimask_output is enabled. Each mask is accompanied by a confidence score that reflects how well it aligns with the model’s internal criteria for consistency and completeness.

The output masks are boolean arrays with the same spatial dimensions as the input image. Each element in the array indicates whether the corresponding pixel belongs to the segmented object. Selecting the mask with the highest score is a straightforward heuristic, but it is important to understand that this choice is not always perfect. Different masks may correspond to different interpretations, such as a full object versus a subpart of it.

The final stage converts the selected mask into an image format suitable for visualization. Since the mask is boolean, it is transformed into an 8-bit grayscale image where foreground pixels are set to white and background pixels to black. This representation is then saved to disk, producing a clear visual separation between the segmented object and the rest of the scene.

From a broader perspective, this code illustrates a shift toward interactive and prompt-driven vision systems. Unlike traditional models that rely on fixed categories, SAM enables a more flexible form of visual understanding where the user defines the task dynamically. This capability makes it particularly useful as a building block in generative pipelines, where precise control over image regions is required for operations such as inpainting, object removal, or content manipulation.

Conceptually, the workflow can be understood as transforming a raw image into a structured representation through three stages: encoding the image into a latent space, conditioning this representation with a user-defined prompt, and decoding the result into a segmentation mask. This approach reflects a broader trend in modern AI systems, where models are designed not just to produce answers, but to collaborate with users through intuitive forms of interaction.

**3. Apply a Visual Mask Overlay**

Task: Highlight a detected region on the original image.

In [5]:
import numpy as np
from PIL import Image

original = Image.open("./data/pic.jpg").convert("RGBA")
mask = Image.open("./data/mask.png").convert("L")  # grayscale mask

# Create a red overlay
overlay = Image.new("RGBA", original.size, (255, 0, 0, 0))
red_layer = Image.new("RGBA", original.size, (255, 0, 0, 120))  # semi-transparent red

# Apply red only where mask is white
overlay.paste(red_layer, mask=mask)

# Composite
result = Image.alpha_composite(original, overlay)
result.convert("RGB").save("./data/masked_overlay.jpg")

This code represents the final stage in a typical computer vision pipeline, where abstract model outputs are transformed into an interpretable visual form. After a segmentation model such as the Segment Anything Model (SAM) produces a mask, that mask by itself is simply a matrix of values. In order to make it meaningful for a human observer, it must be overlaid onto the original image in a way that highlights the selected region without destroying the underlying visual context.

The process begins by loading two separate images: the original photograph and a corresponding mask. The original image is converted into RGBA format, which introduces an alpha channel. This alpha channel controls transparency and is essential for blending multiple image layers together. Without it, the system would not be able to create partially transparent overlays, which are crucial for clear visualization.

The mask is loaded in grayscale mode, meaning that each pixel contains a single intensity value. In this context, the mask functions as a spatial selector. Bright regions, typically white, represent areas of interest, while dark regions represent the background. Conceptually, this mask encodes a function over the image domain that determines where modifications should be applied.

Next, the code constructs two auxiliary images. The first is an empty transparent canvas that matches the dimensions of the original image. The second is a uniformly colored red layer with partial transparency. The alpha value of 120, on a scale from 0 to 255, ensures that the red color is visible but does not completely obscure the image beneath it. This choice reflects an important visualization principle: the goal is to emphasize, not replace, the underlying content.

The key operation occurs when the red layer is pasted onto the transparent overlay using the mask. This step can be understood as a conditional operation applied pixel-wise. Wherever the mask has high intensity values, the red layer is transferred onto the overlay. Where the mask is dark, the overlay remains transparent. In effect, the mask acts as a gate that controls the spatial distribution of the red highlight.

Once the overlay has been constructed, it is combined with the original image using alpha compositing. This operation blends the two images together based on their alpha channels. For each pixel, the final color is computed as a weighted combination of the foreground (the overlay) and the background (the original image). The result is an image where the segmented region is softly highlighted in red, while the rest of the scene remains unchanged.

The final step converts the image back to RGB format and saves it to disk. This conversion removes the alpha channel, producing a standard image that can be viewed or shared without requiring special handling for transparency.

From a conceptual standpoint, this code illustrates how numerical outputs from machine learning models can be integrated into human-centered workflows. The segmentation mask is not inherently meaningful until it is mapped back onto the visual domain in a way that aligns with human perception. By using transparency and color, the code creates a bridge between abstract data and intuitive understanding.

This technique is widely used in applications such as medical imaging, where highlighted regions may correspond to areas of interest in a scan, or in interactive editing tools, where users need immediate visual feedback on selected regions. In the broader context of generative AI systems, such overlays often serve as an intermediate step before further transformations are applied, such as removing objects, replacing regions, or guiding generative models to modify specific parts of an image.

Ultimately, this example demonstrates that effective AI systems are not only about making predictions, but also about presenting those predictions in a way that is clear, interpretable, and actionable.

**4. Object Removal via Inpainting (Stable Diffusion)**

Task: Remove an object and let the model "fill in" what would be behind it.

In [2]:
import torch
print(torch.cuda.is_available())

False


In [6]:
# pip install diffusers transformers accelerate torch Pillow

import torch
from PIL import Image
from diffusers import StableDiffusionInpaintPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=dtype,
)
pipe.to(device)  # GPU strongly recommended

image = Image.open("./data/pic.jpg").convert("RGB").resize((512, 512))
mask = Image.open("./data/mask.png").convert("RGB").resize((512, 512))
# Mask: WHITE = area to fill in, BLACK = keep as-is

result = pipe(
    prompt="background scenery, photorealistic",  # describe what should fill the area
    negative_prompt="cat, dog, person",  # describe what should NOT be generated
    image=image,
    mask_image=mask,
    num_inference_steps=50,
    guidance_scale=7.5,
).images[0]

result.save("./data/inpainted.jpg")

c:\Users\vmelnyk2\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:  14%|█▍        | 1/7 [00:00<00:01,  3.63it/s]An error occurred while trying to fetch C:\Users\vmelnyk2\.cache\huggingface\hub\models--runwayml--stable-diffusion-inpainting\snapshots\8a4288a76071f7280aedbdb3253bdb9e9d5d84bb\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\vmelnyk2\.cache\huggingface\hub\models--runwayml--stable-diffusion-inpainting\snapshots\8a4288a76071f7280aedbdb3253bdb9e9d5d84bb\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  29%|██▊       | 2/7 [00:00<00:01,  4.65it/s]An error occurred while trying to fetch C:\Users\vmelnyk2\.cache\huggingfac

KeyboardInterrupt: 

This code demonstrates one of the most compelling capabilities of modern generative AI: the ability to modify an existing image by synthesizing new content in selected regions. This process is known as inpainting, and it is powered here by a diffusion-based model, specifically Stable Diffusion Inpainting. Unlike traditional computer vision models that only analyze images, this model actively generates new visual information while respecting both the original image and a user-defined constraint.

The process begins with loading a pretrained inpainting pipeline from the diffusers library. When the code calls from_pretrained("runwayml/stable-diffusion-inpainting"), it retrieves a model that has been trained not only to generate images from noise, but also to condition that generation on an existing image and a spatial mask. Internally, this model consists of several components, including a text encoder, a variational autoencoder, and a denoising neural network. Together, these components form a system capable of translating textual descriptions into coherent visual structures.

The model is configured to use half-precision floating point numbers, which reduces memory consumption and improves performance on modern GPUs. The subsequent call to move the model onto a CUDA device is critical, as diffusion models require iterative computation over many steps, making GPU acceleration highly beneficial for practical use.

The next stage prepares the inputs. The original image is loaded and resized to a fixed resolution of 512 by 512 pixels. This resizing is not arbitrary; diffusion models are typically trained on images of a specific size, and deviations from this resolution can lead to degraded results or increased computational cost. Alongside the image, a mask is loaded and resized to the same dimensions. The mask plays a central role in guiding the generation process. It defines a binary spatial constraint where white regions indicate areas to be modified, and black regions indicate areas that must remain unchanged.

When the pipeline is executed, the model receives four key inputs: a textual prompt, the original image, the mask, and a set of generation parameters. The prompt describes what should appear in the masked region. Unlike earlier segmentation or detection models, which interpret existing content, this model uses the prompt to synthesize entirely new content that is consistent with both the surrounding pixels and the semantic description.

Internally, the model operates through a process known as iterative denoising. It begins with a noisy representation of the masked region and gradually refines it over a series of steps. At each step, the model predicts how the noise should be adjusted to better align with the prompt and the visible context of the image. The parameter num_inference_steps controls how many of these refinement iterations are performed. Increasing this value generally improves quality but also increases computation time.

Another important parameter is the guidance scale, which determines how strongly the model adheres to the textual prompt. A higher value forces the generated content to more closely match the description, potentially at the cost of natural blending with the surrounding image. A lower value allows for more flexibility and may produce results that better integrate with the existing scene but are less faithful to the prompt.

The output of the pipeline is a new image in which the masked region has been replaced with generated content. Importantly, the model does not simply fill the area in isolation; it takes into account the surrounding pixels to ensure visual coherence. This means that textures, lighting, and perspective are often preserved in a way that makes the modification appear natural.

Finally, the resulting image is saved to disk. At this stage, the transformation is complete: an input image and a mask have been converted into a modified image that seamlessly integrates newly generated content.

From a conceptual perspective, this code illustrates the culmination of a multi-stage generative workflow. Earlier steps, such as segmentation, define where changes should occur, while the diffusion model determines what those changes should look like. This separation of concerns is a powerful design pattern in modern AI systems, enabling flexible and controllable image editing.

In a broader context, inpainting represents a shift from passive analysis to active creation. The model is no longer limited to describing the world; it participates in reshaping it according to human intent. This capability underlies many contemporary tools in digital art, photo editing, and content creation, where users can remove objects, replace backgrounds, or generate entirely new scenes with minimal manual effort.

**All together:**

In [1]:
import torch
print(torch.__version__)
print(torch.__file__)

2.6.0+cpu
c:\Users\vmelnyk2\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\__init__.py


In [8]:
import numpy as np
import torch
from PIL import Image, ImageFilter
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
from diffusers import StableDiffusionInpaintPipeline

# ── Step 1: Detect with YOLO ──────────────────────────────────────────────────
yolo = YOLO("yolov8n.pt") # Loads pretrained YOLO model
img_path = "./data/pic2.jpg"
results = yolo(img_path)[0] # returns list of results

# Find the first "person" detection
target_box = None # Initialize variable for bounding box
for box in results.boxes:
    if results.names[int(box.cls)] == "person":
        target_box = list(map(int, box.xyxy[0]))  # Extract bounding box [x1, y1, x2, y2]
        break

if target_box is None:
    raise RuntimeError("No person detected in image")

print(f"Detected person at box: {target_box}")

# ── Step 2: Segment with SAM ──────────────────────────────────────────────────
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth") # Loads SAM model:
# "vit_b" = medium-sized transformer
# uses local checkpoint file

sam.to("cuda" if torch.cuda.is_available() else "cpu") # Moves model to GPU if available
predictor = SamPredictor(sam) # Wraps model with helper class for easier usage.

img_np = np.array(Image.open(img_path).convert("RGB")) # Loads image and converts to: RGB format, numpy array
predictor.set_image(img_np) # Preprocesses image: computes image embeddings and moves to same device as model

masks, scores, _ = predictor.predict(
    box=np.array(target_box),   # SAM accepts YOLO boxes directly!
    multimask_output=True,
)
# SAM:
# takes bounding box as input prompt
# returns:
# masks -> possible segmentations
# scores -> confidence for each mask

best_mask = masks[np.argmax(scores)]  # (H, W) bool array. Selects best mask based on confidence score

# ── Step 3: Prepare mask for inpainting ──────────────────────────────────────
mask_pil = Image.fromarray((best_mask * 255).astype(np.uint8), mode="L") # Converts mask to image
# True -> 255 (white)

# Dilate mask slightly so edges are cleanly covered
mask_dilated = mask_pil.filter(ImageFilter.MaxFilter(size=15))

# ── Step 4: Inpaint with Stable Diffusion ────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=dtype,
    # use_safetensors=True,
)
pipe.to(device)

original = Image.open(img_path).convert("RGB").resize((512, 512))
mask_resized = mask_dilated.resize((512, 512))

output = pipe(
    prompt="background, photorealistic, high quality", # Tells model what to generate
    negative_prompt="person, human, body", # Tells model what to avoid generating
    image=original,
    mask_image=mask_resized,
    num_inference_steps=50,
    guidance_scale=8.0,
).images[0]

output.save("./data/person_removed.jpg")
print("Done! Saved to person_removed.jpg")


image 1/1 c:\personal\genai\gen-ai\new\data\pic2.jpg: 448x640 1 person, 157.3ms
Speed: 5.1ms preprocess, 157.3ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
Detected person at box: [326, 81, 2511, 2532]


C:\Users\vmelnyk2\AppData\Local\Temp\ipykernel_22288\2003247633.py:49: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  mask_pil = Image.fromarray((best_mask * 255).astype(np.uint8), mode="L") # Converts mask to image
Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\vmelnyk2\.cache\huggingface\hub\models--runwayml--stable-diffusion-inpainting\snapshots\8a4288a76071f7280aedbdb3253bdb9e9d5d84bb\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\vmelnyk2\.cache\huggingface\hub\models--runwayml--stable-diffusion-inpainting\snapshots\8a4288a76071f7280aedbdb3253bdb9e9d5d84bb\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  29%|██▊       | 2/7 [00:00<00:00, 13.17it/s]An error occurred while trying to fetch C:\Users\vmelnyk2\.cache\huggingface\hub\models--runwaym

Done! Saved to person_removed.jpg


**Count objects in image**

In [ ]:
from ultralytics import YOLO
from collections import Counter

model = YOLO("yolov8n.pt")

results = model("./data/pic.jpg")[0]

counts = Counter()

for box in results.boxes:
    cls_name = results.names[int(box.cls)]
    counts[cls_name] += 1

print("Detected objects:")
for obj, count in counts.items():
    print(f"{obj}: {count}")


image 1/1 c:\personal\genai\gen-ai\new\data\pic2.jpg: 448x640 1 person, 124.5ms
Speed: 4.6ms preprocess, 124.5ms inference, 1.8ms postprocess per image at shape (1, 3, 448, 640)
Detected objects:
person: 1


**Put sunglasses on every detected person**

In [5]:
from ultralytics import YOLO
from PIL import Image

model = YOLO("yolov8n.pt")

img = Image.open("./data/pic2.jpg")
overlay = Image.open("./data/glasses.png").convert("RGBA")

results = model("./data/pic2.jpg")[0]

for box in results.boxes:
    if results.names[int(box.cls)] == "person":
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        w, h = x2 - x1, y2 - y1

        glasses_resized = overlay.resize((w, h//3))
        img.paste(glasses_resized, (x1, y1), glasses_resized)

img.save("./data/fun_output.jpg")


image 1/1 c:\personal\genai\gen-ai\new\data\pic2.jpg: 448x640 1 person, 161.7ms
Speed: 4.6ms preprocess, 161.7ms inference, 2.0ms postprocess per image at shape (1, 3, 448, 640)


**Dog only**

In [9]:
from ultralytics import YOLO
from PIL import Image, ImageDraw

model = YOLO("yolov8n.pt")

img = Image.open("./data/pic.jpg")
draw = ImageDraw.Draw(img)

results = model("./data/pic.jpg")[0]

for box in results.boxes:
    cls = results.names[int(box.cls)]
    if cls == "dog":
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        draw.rectangle([x1, y1, x2, y2], outline="green", width=3)

img.save("./data/dogs_only.jpg")


image 1/1 c:\personal\genai\gen-ai\new\data\pic.jpg: 384x640 1 bird, 1 cat, 1 dog, 115.8ms
Speed: 5.5ms preprocess, 115.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


**Interaction!**

In [7]:
import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    results = model(frame)[0]

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = results.names[int(box.cls)]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, cls, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    cv2.imshow("YOLO Demo", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 1 potted plant, 236.1ms
Speed: 20.3ms preprocess, 236.1ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 305.8ms
Speed: 10.0ms preprocess, 305.8ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 251.6ms
Speed: 5.1ms preprocess, 251.6ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 234.1ms
Speed: 3.3ms preprocess, 234.1ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 359.3ms
Speed: 4.1ms preprocess, 359.3ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 271.2ms
Speed: 5.6ms preprocess, 271.2ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 226.2ms
Speed: 3.8ms preprocess, 226.2ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 255.7ms
Speed: 2.7ms preprocess, 255.

KeyboardInterrupt: 